In [1]:
!pip install -q -U trl peft bitsandbytes accelerate datasets

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import glob
import json
import re
import pandas as pd
import torch
from datasets import Dataset
from peft import LoraConfig, PeftModel
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import GRPOConfig, GRPOTrainer, SFTConfig, SFTTrainer

# Đọc 3 tập dữ liệu chuẩn
sft_files = glob.glob("/kaggle/input/**/D_SFT.parquet", recursive=True)
grpo_files = glob.glob("/kaggle/input/**/D_GRPO.parquet", recursive=True)
test_files = glob.glob("/kaggle/input/**/test_9600.parquet", recursive=True)
assert len(sft_files) > 0 and len(grpo_files) > 0 and len(test_files) > 0, "Thiếu dataset input (read-splits-v1 hoặc read-distilled-v1)!"

df_sft = pd.read_parquet(sft_files[0])
df_grpo = pd.read_parquet(grpo_files[0])
df_test = pd.read_parquet(test_files[0])

print(f"D_SFT: {len(df_sft):,} | D_GRPO pool: {len(df_grpo):,} | Test: {len(df_test):,}", flush=True)

SYSTEM_PROMPT = (
    "You are an expert in textual forensics. Decide whether the given text is AI-generated or human-written. "
    "Reason step by step considering style, lexical diversity, sentence rhythm, structural symmetry, and concrete specificity. "
    "Use transition words such as 'First', 'Second', 'Moreover' and end your reasoning with your conclusion. "
    "Return STRICT JSON ONLY with exact keys: "
    '{"rationale": "3-5 concise text-specific sentences", "verdict": "AI" or "HUMAN"}.'
)

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side="right")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
D_SFT: 2,951 | D_GRPO pool: 41,049 | Test: 9,600


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [2]:
print("Các cột hiện có trong D_SFT:", df_sft.columns.tolist(), flush=True)

def normalize_sft_df(df):
    df = df.copy()
    rat_candidates = ["rationale", "teacher_rationale", "reasoning", "explanation", "cot"]
    rat_col = next((c for c in rat_candidates if c in df.columns), None)
    df["rationale_clean"] = df[rat_col].astype(str).str.strip()

    if "verdict" in df.columns:
        df["verdict_clean"] = df["verdict"].astype(str).str.strip().str.upper()
    else:
        df["verdict_clean"] = df["label"].apply(lambda x: "AI" if str(x).strip().upper() == "AI" else "HUMAN")

    df = df[df["rationale_clean"].str.len() >= 20].reset_index(drop=True)
    return df

df_sft_clean = normalize_sft_df(df_sft)
print(f"Số mẫu D_SFT hợp lệ sau chuẩn hóa: {len(df_sft_clean):,}", flush=True)

def format_sft_example(row):
    text_clip = " ".join(str(row["text"]).split()[:800])
    target_json = json.dumps(
        {"rationale": row["rationale_clean"], "verdict": row["verdict_clean"]},
        ensure_ascii=False,
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Analyze this text:\n\n{text_clip}"},
        {"role": "assistant", "content": target_json},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

ds_sft = Dataset.from_pandas(df_sft_clean).map(format_sft_example, remove_columns=df_sft_clean.columns.tolist())

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map={"": 0},
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

sft_out_dir = "/kaggle/working/student_sft_adapter"
sft_args = SFTConfig(
    output_dir=sft_out_dir,
    dataset_text_field="text",
    max_length=1536,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=2,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    fp16=False,
    bf16=False,
    logging_steps=25,
    save_strategy="no",
    report_to="none",
)

sft_trainer = SFTTrainer(
    model=base_model,
    train_dataset=ds_sft,
    peft_config=lora_config,
    processing_class=tokenizer,
    args=sft_args,
)

sft_trainer.train()
sft_trainer.model.save_pretrained(sft_out_dir)
tokenizer.save_pretrained(sft_out_dir)
print("Đã lưu xong adapter Pha 1 (Student-SFT) tại:", sft_out_dir, flush=True)

del sft_trainer, base_model
torch.cuda.empty_cache()

Các cột hiện có trong D_SFT: ['sample_id', 'text', 'label', 'domain', 'source', 'raw_response', 'reasoning', 'pred_label', 'gold_label', 'target_response']
Số mẫu D_SFT hợp lệ sau chuẩn hóa: 2,951


Map:   0%|          | 0/2951 [00:00<?, ? examples/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

... (5 outputs left out)

Truncating train dataset:   0%|          | 0/2951 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/2951 [00:00<?, ? examples/s]

Step,Training Loss
25,2.270081
50,1.330635
75,1.422325
100,1.405104
125,1.352694
150,1.365261
175,1.490749
200,1.346372
225,1.318805
250,1.333143


Đã lưu xong adapter Pha 1 (Student-SFT) tại: /kaggle/working/student_sft_adapter


In [3]:
df_grpo_sub = (
    df_grpo.groupby(["domain", "label"], group_keys=False)
    .apply(lambda g: g.sample(n=min(50, len(g)), random_state=42))
    .sample(frac=1.0, random_state=42)
    .reset_index(drop=True)
)

tokenizer_left = AutoTokenizer.from_pretrained(MODEL_ID, padding_side="left")
if tokenizer_left.pad_token is None:
    tokenizer_left.pad_token = tokenizer_left.eos_token

def format_grpo_prompt(row):
    text_clip = " ".join(str(row["text"]).split()[:800])
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Analyze this text:\n\n{text_clip}"},
    ]
    return {
        "prompt": tokenizer_left.apply_chat_template(messages, tokenize=False, add_generation_prompt=True),
        "ground_truth": "AI" if str(row["label"]).strip().upper() == "AI" else "HUMAN",
    }

ds_grpo = Dataset.from_pandas(df_grpo_sub).map(format_grpo_prompt, remove_columns=df_grpo_sub.columns.tolist())

def parse_prediction(text):
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        return None, None
    try:
        obj = json.loads(match.group(0))
        rat = str(obj.get("rationale", "")).strip()
        ver = str(obj.get("verdict", "")).strip().upper()
        if ver in {"AI", "HUMAN"} and len(rat) >= 30:
            return rat, ver
    except Exception:
        pass
    return None, None

def format_reward_func(completions, **kwargs):
    rewards = []
    for comp in completions:
        rat, ver = parse_prediction(comp)
        rewards.append(0.5 if ver is not None else -0.5)
    return rewards

def accuracy_reward_func(completions, ground_truth, **kwargs):
    rewards = []
    for comp, gt in zip(completions, ground_truth):
        _, ver = parse_prediction(comp)
        if ver is None:
            rewards.append(0.0)
        elif ver == gt:
            rewards.append(1.5)
        else:
            rewards.append(0.0)
    return rewards

sft_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map={"": 0},
)
model_grpo = PeftModel.from_pretrained(sft_base, sft_out_dir, is_trainable=True)

grpo_out_dir = "/kaggle/working/student_grpo_adapter"
grpo_args = GRPOConfig(
    output_dir=grpo_out_dir,
    learning_rate=5e-6,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_prompt_length=1100,
    max_completion_length=260,
    num_train_epochs=1,
    fp16=False,
    bf16=False,
    logging_steps=20,
    save_strategy="no",
    report_to="none",
)

grpo_trainer = GRPOTrainer(
    model=model_grpo,
    reward_funcs=[format_reward_func, accuracy_reward_func],
    args=grpo_args,
    train_dataset=ds_grpo,
    processing_class=tokenizer_left,
)

grpo_trainer.train()
grpo_trainer.model.save_pretrained(grpo_out_dir)
tokenizer_left.save_pretrained(grpo_out_dir)
print("Đã lưu xong mô hình chính (Student-GRPO) tại:", grpo_out_dir, flush=True)

# ĐÁNH GIÁ TRÊN TOÀN BỘ TẬP TEST 9,600 MẪU
grpo_trainer.model.eval()
eval_out_path = "/kaggle/working/student_grpo_test_predictions.parquet"
results = []
BATCH_EVAL = 32

sub_col = "sub_source" if "sub_source" in df_test.columns else ("source" if "source" in df_test.columns else None)
src_ds_col = "source_dataset" if "source_dataset" in df_test.columns else None

for start in tqdm(range(0, len(df_test), BATCH_EVAL), desc="Evaluating on Test 9,600"):
    batch_df = df_test.iloc[start : start + BATCH_EVAL]
    prompts = []
    for _, row in batch_df.iterrows():
        text_clip = " ".join(str(row["text"]).split()[:800])
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Analyze this text:\n\n{text_clip}"},
        ]
        prompts.append(tokenizer_left.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

    inputs = tokenizer_left(prompts, return_tensors="pt", padding=True, truncation=True, max_length=1200).to("cuda:0")
    with torch.no_grad():
        outputs = grpo_trainer.model.generate(
            **inputs,
            max_new_tokens=260,
            do_sample=False,
            pad_token_id=tokenizer_left.pad_token_id,
        )

    gen_tokens = outputs[:, inputs["input_ids"].shape[1] :]
    decoded = tokenizer_left.batch_decode(gen_tokens, skip_special_tokens=True)

    for (_, row), raw_out in zip(batch_df.iterrows(), decoded):
        rat, ver = parse_prediction(raw_out)
        results.append({
            "sample_id": int(row["sample_id"]),
            "domain": row["domain"],
            "sub_source": row[sub_col] if sub_col else "",
            "source_dataset": row[src_ds_col] if src_ds_col else "",
            "label": row["label"],
            "raw_response": raw_out.strip(),
            "rationale": rat if rat is not None else "",
            "verdict": ver if ver is not None else "INVALID",
            "pred_label": "AI" if ver == "AI" else ("Human" if ver == "HUMAN" else "INVALID"),
        })

df_eval = pd.DataFrame(results)
df_eval.to_parquet(eval_out_path, index=False)

valid_df = df_eval[df_eval["pred_label"] != "INVALID"]
acc_all = (df_eval["pred_label"] == df_eval["label"]).mean()
acc_valid = (valid_df["pred_label"] == valid_df["label"]).mean() if len(valid_df) > 0 else 0.0
fer = 1.0 - (len(valid_df) / len(df_eval))

print("\n=== KẾT QUẢ MÔ HÌNH CHÍNH (STUDENT SFT+GRPO) TRÊN TEST 9,600 ===", flush=True)
print(f"Accuracy toàn bộ (tính cả lỗi format) : {acc_all:.4f}", flush=True)
print(f"Accuracy trên các mẫu hợp lệ         : {acc_valid:.4f}", flush=True)
print(f"Format Error Rate (FER)              : {fer:.4f}", flush=True)

/tmp/ipykernel_23/473827136.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=min(50, len(g)), random_state=42))


Map:   0%|          | 0/1100 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

TypeError: GRPOConfig.__init__() got an unexpected keyword argument 'max_prompt_length'